# Bitcoin Price Forecasting with Heterogeneous Financial Indicators

## Problem Statement

**Research goal:** Predicting Bitcoin price using heterogeneous financial, macroeconomic, crypto-market and technical indicators, and comparing classical time-series, linear, tree-based, ensemble and neural forecasting methods.

The project is a supervised forecasting/data mining task. For each observation frequency, the feature vector available at time `t` is used to predict the Bitcoin price at `t+1`. This one-step-ahead framing avoids using same-period target information as if it were known before the forecast.

The target variable is `BTC`. The explanatory variables include crypto-market assets, equity/ETF proxies, commodities, FX rates, technical indicators, calendar variables, macroeconomic indicators, blockchain/market-sentiment variables, and lagged or rolling BTC features.


## Setup

The notebook uses frozen CSV datasets created by the existing data collection scripts. No external API call is required to reproduce the report.


In [ ]:
import os
from pathlib import Path
import warnings
from collections import OrderedDict

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp/matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/private/tmp')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
Path(os.environ['XDG_CACHE_HOME']).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as pltЗ
import seaborn as sns

from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)


## Data Collection

The project uses three consolidated datasets from the existing BTC forecasting project:

- `new_day_df.csv`: daily observations.
- `new_hour_df.csv`: hourly observations.
- `new_5min_df.csv`: 5-minute observations.

The raw collection and merging logic is kept in `get_data.py`. The report uses the frozen CSV files so that all experiments are reproducible without network access. Every dataset used here has well above 50 objects and 7-10 features.


In [ ]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'new_day_df.csv').exists():
    PROJECT_DIR = Path(__file__).parent if '__file__' in dir() else PROJECT_DIR

DATA_CONFIG = OrderedDict({
    'daily': {
        'file': 'new_day_df.csv',
        'date_col': 'Date',
        'test_size': 216,
        'horizon': 1,
        'ma_window': 7,
        'seasonal_period': 7,
        'max_features': 30,
    },
    'hourly': {
        'file': 'new_hour_df.csv',
        'date_col': 'Datetime',
        'test_size': 0.20,
        'horizon': 1,
        'ma_window': 24,
        'seasonal_period': 24,
        'max_features': 25,
    },
    '5min': {
        'file': 'new_5min_df.csv',
        'date_col': 'Datetime',
        'test_size': 0.20,
        'horizon': 1,
        'ma_window': 12,
        'seasonal_period': 288,
        'max_features': 20,
    },
})

def load_dataset(file_name, date_col):
    df = pd.read_csv(PROJECT_DIR / file_name, parse_dates=[date_col])
    df = df.sort_values(date_col).set_index(date_col)
    df = df.replace([np.inf, -np.inf], np.nan)
    return df

datasets = {
    frequency: load_dataset(cfg['file'], cfg['date_col'])
    for frequency, cfg in DATA_CONFIG.items()
}

for frequency, df in datasets.items():
    print(f'{frequency}: {df.shape[0]} rows, {df.shape[1]} columns, {df.index.min()} -> {df.index.max()}')


## Dataset Summary with Basic Statistics and Plots

This section checks dataset size, observation periods, feature groups, missing values, basic target statistics, and target distribution.


In [ ]:
CRYPTO_COLS = {
    'BTC', 'ETH', 'Bitcoin_Cash', 'Cardano', 'Dogecoin', 'Ethereum_Classic',
    'Chainlink', 'Litecoin', 'TRON', 'Tether', 'Stellar', 'XRP', 'BTC_Supply',
    'Feer_Greed_value'
}
MACRO_COLS = {
    'Unemployment_Rate', 'Non_Farm_Payrolls', 'Initial_Jobless_Claims', 'CPI',
    'PCE_Price_Index', 'Consumer_Confidence_Index', 'Retail_Sales', 'Housing_Starts',
    'Building_Permits', 'M2_Money_Supply', 'Real_Interest_Rate', 'Fed_Funds_Rate',
    'Reverse_Repo_Volume', 'Debt', 'Real_GDP'
}
TECHNICAL_TOKENS = (
    'RSI', 'MACD', 'ATR', 'OBV', 'SMA', 'EMA', 'TEMA', 'KAMA', 'HMA', 'ALMA',
    'T3', 'SSF', 'VIDYA', 'ZL_EMA', 'BTC_lag', 'BTC_mean', 'BTC_std',
    'Open', 'High', 'Low', 'Close', 'Volume', 'Taker buy'
)
CALENDAR_COLS = {'day', 'month', 'quarter', 'year', 'day_of_week', 'hour', 'minute', 'is_holiday'}
FX_TOKENS = ('USD/', 'EUR/', 'GBP/', 'AUD/')
COMMODITY_COLS = {'Gold', 'Silver', 'Copper', 'Platinum', 'Brent_Oil', 'WTI', 'Brent', 'Natural_Gas'}

def feature_group(col):
    if col in CALENDAR_COLS:
        return 'calendar'
    if col in MACRO_COLS:
        return 'macroeconomic'
    if col in CRYPTO_COLS:
        return 'crypto-market'
    if col in COMMODITY_COLS:
        return 'commodities'
    if any(token in col for token in TECHNICAL_TOKENS):
        return 'technical/lagged BTC'
    if any(token in col for token in FX_TOKENS):
        return 'FX'
    return 'equity/ETF/market index'

summary_rows = []
group_rows = []
for frequency, df in datasets.items():
    target = df['BTC']
    missing_by_col = df.isna().sum()
    summary_rows.append({
        'frequency': frequency,
        'objects': len(df),
        'features_excluding_target': df.shape[1] - 1,
        'start': df.index.min(),
        'end': df.index.max(),
        'missing_values_total': int(missing_by_col.sum()),
        'columns_with_missing': int((missing_by_col > 0).sum()),
        'BTC_mean': target.mean(),
        'BTC_std': target.std(),
        'BTC_min': target.min(),
        'BTC_median': target.median(),
        'BTC_max': target.max(),
    })
    feature_groups = pd.Series([feature_group(col) for col in df.columns if col != 'BTC']).value_counts()
    for group, count in feature_groups.items():
        group_rows.append({'frequency': frequency, 'feature_group': group, 'count': int(count)})

summary_df = pd.DataFrame(summary_rows)
feature_group_df = pd.DataFrame(group_rows).sort_values(['frequency', 'feature_group'])

display(summary_df)
display(feature_group_df.pivot_table(index='feature_group', columns='frequency', values='count', fill_value=0).astype(int))


In [ ]:
basic_stats = pd.concat(
    {frequency: df['BTC'].describe() for frequency, df in datasets.items()},
    axis=1
).T
basic_stats['skew'] = [datasets[f]['BTC'].skew() for f in basic_stats.index]
basic_stats['kurtosis'] = [datasets[f]['BTC'].kurtosis() for f in basic_stats.index]
display(basic_stats.round(2))


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
for row, (frequency, df) in enumerate(datasets.items()):
    # Левая колонка: уровень цены в лог-шкале — виден мультипликативный тренд/режимы.
    df['BTC'].plot(ax=axes[row, 0], color='tab:blue', linewidth=1.0, logy=True)
    axes[row, 0].set_title(f'{frequency.title()} BTC price (log scale)')
    axes[row, 0].set_xlabel('')
    # Правая колонка: распределение лог-доходностей — целевая переменная моделей.
    # В отличие от уровня цены, доходности (около)стационарны и центрированы у нуля,
    # поэтому их распределение информативно (тяжёлые хвосты, эксцесс); гистограмма
    # самого уровня была бы лишь мультимодальным отражением ценовых режимов.
    logret = np.log(df['BTC']).diff().dropna()
    sns.histplot(logret, bins=60, kde=True, ax=axes[row, 1], color='tab:orange')
    axes[row, 1].set_title(f'{frequency.title()} log-return distribution '
                           f'(std={logret.std():.4f}, kurt={logret.kurt():.1f})')
    axes[row, 1].set_xlabel('log return')
plt.tight_layout()
plt.show()


### Missing Values and Outliers

The consolidated files have no missing values. The modeling pipeline still includes median imputation to make the workflow robust to future data refreshes. Outliers are handled inside the training pipeline by clipping numeric features to train-fitted 1st and 99th percentiles before robust scaling.


In [ ]:
def iqr_outlier_count(frame):
    numeric = frame.select_dtypes(include=[np.number])
    q1 = numeric.quantile(0.25)
    q3 = numeric.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = numeric.lt(lower, axis=1) | numeric.gt(upper, axis=1)
    return mask.sum().sort_values(ascending=False)

missing_table = pd.DataFrame({
    frequency: df.isna().sum() for frequency, df in datasets.items()
}).fillna(0).astype(int)

display(missing_table.sum().rename('missing_values_by_frequency').to_frame())

outlier_rows = []
for frequency, df in datasets.items():
    counts = iqr_outlier_count(df).head(10)
    for feature, count in counts.items():
        outlier_rows.append({'frequency': frequency, 'feature': feature, 'IQR_outlier_count': int(count)})

display(pd.DataFrame(outlier_rows))


## Methodology

The complete pipeline is:

1. Load frozen datasets (Daily, Hourly, 5-minute) and validate size, period, missing values, and feature groups.
2. Convert the task to one-step-ahead forecasting: features at `t` predict `BTC` at `t+1`.
3. Use time-based train/test splits only. No shuffling is used.
4. Use a fixed shortlist of the strongest train-period predictors per frequency (`TOP_FEATURES`), pre-computed once from train-only absolute correlations with the next-period target.
5. Fit preprocessing only on training data: median imputation, 1/99 percentile clipping, robust scaling.
6. Train two baselines (Naive last value, Moving average), four Gradient Boosting variants (HistGradientBoosting, LightGBM, XGBoost, CatBoost), and four neural sequence architectures (LSTM, GRU, Stacked LSTM, CNN-LSTM, all built with `CryptoNet`).
7. Compare models by MAE, RMSE, SMAPE, MASE, and train/test gap; select the best Gradient Boosting model per frequency (`best_gb`).
8. Interpret `best_gb` with permutation importance and SHAP, and the NN models with gradient-based feature attribution.
9. Analyze residuals and compare model behavior during normal periods versus sharp BTC growth/drop periods.

This setup directly targets the thesis comparison: **Gradient Boosting ensembles vs. Neural Networks**, evaluated at Daily, Hourly, and 5-minute granularities, with feature importance and shock-period error analysis as the basis for interpretation.


In [ ]:
import sys
sys.path.append(str(PROJECT_DIR))

from ml_models import (
    CryptoNetRegressor, _sample_crypto_hparams, run_optuna_study,
    set_global_seed, DEVICE,
)
set_global_seed(RANDOM_STATE)


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    ratio = np.divide(np.abs(y_true - y_pred), denominator, out=np.zeros_like(denominator), where=denominator != 0)
    return float(np.mean(ratio))


def mase(y_true, y_pred, y_train):
    y_train = np.asarray(y_train, dtype=float)
    scale = np.mean(np.abs(np.diff(y_train)))
    if scale == 0 or np.isnan(scale):
        return np.nan
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / scale)


def regression_metrics(y_true, y_pred, y_train):
    frame = pd.concat([
        pd.Series(y_true, name='true'),
        pd.Series(y_pred, name='pred')
    ], axis=1).dropna()
    if frame.empty:
        return {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan, 'DA': np.nan}
    return {
        'MAE': mean_absolute_error(frame['true'], frame['pred']),
        'RMSE': np.sqrt(mean_squared_error(frame['true'], frame['pred'])),
        'MASE': mase(frame['true'], frame['pred'], y_train),
        # Directional Accuracy: доля совпадений знака (для таргета-доходности)
        'DA': float(np.mean((frame['true'].values > 0) == (frame['pred'].values > 0))),
    }


class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.nanquantile(X, self.lower, axis=0)
        self.upper_bounds_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_bounds_, self.upper_bounds_)


def make_preprocessed_model(model, scale=True):
    steps = [
        ('imputer', SimpleImputer(strategy='median')),
        ('clipper', QuantileClipper(lower=0.01, upper=0.99)),
    ]
    if scale:
        steps.append(('scaler', RobustScaler()))
    steps.append(('model', model))
    return Pipeline(steps)


SMAPE_SCORER = make_scorer(smape, greater_is_better=False)


In [ ]:
# Hardcoded configuration: feature shortlists and model hyperparameters.
#
# TOP_FEATURES — пул признаков-кандидатов для пофолдового top-k отбора (НЕ
# фиксированный шорт-лист). Пять прежних признаков уровня цены (BTC_lag*, BTC_mean5)
# заменены на признаки ДОХОДНОСТИ (ret_lag*, ret_mean5, ret_std10): таргет теперь —
# лог-доходность, и предикторы тоже должны быть стационарными (импульс/волатильность),
# иначе модель ключевается на нестационарном уровне цены. Все ret_*-признаки строго
# лагированы в prep_returns. Остальной пул — экзогенные ряды (уже лагированы в
# get_data) и RSI.
TOP_FEATURES = {
    'daily': [
        # признаки доходности (импульс / волатильность)
        'ret_lag1', 'ret_lag2', 'ret_lag5', 'ret_lag10',
        'ret_mean5', 'ret_mean10', 'ret_mean20', 'ret_std5', 'ret_std10', 'ret_std20',
        # тренд / возврат к среднему + нормированные технические + объём (OBV)
        'p_sma10', 'p_sma20', 'ema_cross', 'RSI', 'macd_norm', 'atr_pct', 'obv_chg',
        # настроения / риск-режим
        'fg_value', 'fg_chg', 'VIX',
        # экзогенные активы / индексы
        'SPDR_Communications', 'TRON', 'S&P_500', 'NASDAQ_100', 'MicroStrategy',
        'Gold_Future', 'XRP', 'ETH', 'Meta', 'NVIDIA',
    ],
    'hourly': [
        'ret_lag1', 'ret_lag2', 'ret_lag5', 'ret_lag10',
        'ret_mean5', 'ret_mean10', 'ret_mean20', 'ret_std5', 'ret_std10', 'ret_std20',
        'p_sma10', 'p_sma20', 'ema_cross', 'RSI', 'macd_norm', 'atr_pct', 'obv_chg', 'VIX',
        'XRP', 'TRON', 'Stellar', 'MicroStrategy', 'Litecoin', 'Meta',
        'SPDR_Financials', 'Amazon', 'Cardano', 'Chainlink',
    ],
    '5min': [
        'ret_lag1', 'ret_lag2', 'ret_lag5', 'ret_lag10',
        'ret_mean5', 'ret_mean10', 'ret_mean20', 'ret_std5', 'ret_std10', 'ret_std20',
        'p_sma10', 'p_sma20', 'ema_cross', 'RSI', 'macd_norm', 'atr_pct', 'obv_chg', 'VIX',
        'MicroStrategy', 'Cardano', 'Dogecoin', 'Chainlink', 'Litecoin', 'XRP',
        'Amazon', 'Google', 'Ethereum_Classic',
    ],
}

# Gradient boosting hyperparameters (structural params fixed; lr/n_est tuned per fold by Optuna).
GB_HYPERPARAMS = {
    'HistGradientBoosting': dict(
        max_iter=260, learning_rate=0.05, l2_regularization=0.05, random_state=RANDOM_STATE,
    ),
    'LightGBM': dict(
        n_estimators=320, learning_rate=0.03, num_leaves=31, min_child_samples=20,
        subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    ),
    'XGBoost': dict(
        objective='reg:squarederror', n_estimators=320, learning_rate=0.03, max_depth=3,
        subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'CatBoost': dict(
        iterations=320, learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
        random_state=RANDOM_STATE, verbose=False,
    ),
}

# Sequence-model (CryptoNet) per-frequency settings.
NN_CONFIG = {
    'daily':  {'window_size': 10, 'epochs': 30, 'batch_size': 16},
    'hourly': {'window_size': 24, 'epochs': 25, 'batch_size': 32},
    '5min':   {'window_size': 24, 'epochs': 25, 'batch_size': 64},
}
NN_ARCHITECTURES = ['LSTM', 'GRU', 'StackedLSTM', 'CNN_LSTM']
RUN_NN_CLF = True   # доп. дорожка: взвешенная классификация знака доходности (BCE)
NN_OPTUNA_TRIALS = 5  # retained for reference; the experiment runs through walk-forward

# Walk-forward: КОРОТКИЕ скользящие окна с частым переобучением (борьба с дрейфом).
# train подобран так, чтобы у нейросетей оставалось достаточно окон-примеров, а test —
# короткий следующий блок. Optuna переобучается раз в RETUNE_EVERY фолдов (между ними —
# только refit), иначе сотни фолдов неподъёмны по времени.
#   daily : train 180 (~6 мес) / test 21 (~1 мес)   -> ~130 фолдов, OOF ~94%
#   hourly: train 504 (~3 нед) / test 48 (~2 дня)   -> ~425 фолдов, OOF ~98%
#   5min  : train 1440 (~5 дн) / test 288 (~1 день) -> ~52 фолда,  OOF ~92%
SPLIT_CONFIG = {
    'daily':  {'type': 'rolling', 'n_splits': None, 'train_size': 180,  'test_size': 21},
    'hourly': {'type': 'rolling', 'n_splits': None, 'train_size': 504,  'test_size': 48},
    '5min':   {'type': 'rolling', 'n_splits': None, 'train_size': 1440, 'test_size': 288},
}
RETUNE_EVERY = 20          # Optuna re-tunes every 20 folds; refit-only otherwise
WF_OPTUNA_TRIALS = 25      # GB: Optuna trials per (re)tune (расширенное HP-пространство, T3.2)
WF_NN_OPTUNA_TRIALS = 8    # NN: больше триалов под расширенную сетку (lr, weight_decay, размеры)
WF_TOP_K = 20   # features retained per fold via |Pearson| ranking on train fold only

# Annualisation constant for Sharpe ratio in backtest_summary
PERIODS_PER_YEAR = {'daily': 252, 'hourly': 8760, '5min': 105120}


In [ ]:
# Подготовка датасетов под прогноз ДОХОДНОСТИ (а не уровня цены).
#
# Мотивация: уровень цены BTC нестационарен и трендовый. (1) Бустинги не
# экстраполируют за пределы диапазона таргета на трейне -> на растущем ряду они
# систематически недооценивают в поздних фолдах, что делает сравнение GB vs NN
# некорректным; (2) на уровне SMAPE вырождается до персистентности, а MASE
# завышается из-за смены ценового режима между трейном и тестом. Переход к
# лог-доходности r_t = log(P_t) - log(P_{t-1}) делает таргет (почти) стационарным:
# деревьям не нужно экстраполировать, метрики (включая Directional Accuracy) обретают
# смысл, а цена восстанавливается как P_t = P_{t-1} * exp(r_t).

RET_LAGS = [1, 2, 5, 10]
# Контемпоранные BTC-производные из CSV посчитаны по "текущему" бару t и содержат
# BTC_t -> при прогнозе r_t это утечка. Сдвигаем их на 1 бар. Экзогенные ряды уже
# лагированы в get_data, календарные признаки известны заранее -> их НЕ трогаем.
CONTEMP_BTC_COLS = ['RSI', 'MACD', 'ATR', 'OBV', 'BTC_mean5', 'BTC_std10']


def prep_returns(df, feature_cols):
    """Фрейм для walk-forward с таргетом-лог-доходностью.

    Возвращает копию df, в которой:
      * BTC        - исходный уровень цены (для восстановления цены и графиков);
      * BTC_logret - целевая переменная r_t = log(P_t) - log(P_{t-1});
      * ret_lag*, ret_mean5, ret_std10 - строго лагированные (>= t-1) признаки
        доходности (импульс/волатильность), не содержащие r_t;
      * контемпоранные BTC-производные (RSI/MACD/ATR/OBV/BTC_mean5/BTC_std10)
        сдвинуты на 1 бар, чтобы исключить BTC_t.
    Признаки строятся уже с нужным лагом, поэтому в самом walk-forward
    дополнительный shift НЕ применяется (единый источник истины о зазоре).
    """
    out = df.copy()
    price = out['BTC'].astype(float)
    ret = np.log(price).replace([np.inf, -np.inf], np.nan).diff()
    out['BTC_logret'] = ret
    for L in RET_LAGS:
        out[f'ret_lag{L}'] = ret.shift(L)
    # Скользящие средние и волатильность доходностей на нескольких горизонтах
    # (импульс / режим волатильности); все строго лагированы (ret.shift(1)).
    for w in (5, 10, 20):
        out[f'ret_mean{w}'] = ret.shift(1).rolling(w).mean()
        out[f'ret_std{w}'] = ret.shift(1).rolling(w).std()
    # Цена относительно собственных скользящих средних (тренд / возврат к среднему)
    # и EMA-кроссовер; используется лагированная цена P_{t-1}.
    p_lag = price.shift(1)
    for w in (10, 20):
        out[f'p_sma{w}'] = p_lag / p_lag.rolling(w).mean() - 1
    out['ema_cross'] = (p_lag.ewm(span=5, adjust=False).mean()
                        / p_lag.ewm(span=20, adjust=False).mean() - 1)
    # Нормированные технические индикаторы (без зависимости от уровня цены) и
    # объёмный момент через OBV (сырого объёма в CSV нет — OBV суммирует объём).
    if 'MACD' in out.columns: out['macd_norm'] = (out['MACD'] / price).shift(1)
    if 'ATR' in out.columns: out['atr_pct'] = (out['ATR'] / price).shift(1)
    if 'OBV' in out.columns:
        _obv_d = out['OBV'].diff()
        # знаковый лог сжимает огромный объёмный масштаб OBV.diff() (до ~1e11) к ~[-27,27],
        # сохраняя порядок; иначе скейлер NN получает гигантские входы и обучение расходится.
        out['obv_chg'] = (np.sign(_obv_d) * np.log1p(_obv_d.abs())).shift(1)
    # Индекс страха и жадности — присутствует только в дневных данных.
    if 'Feer_Greed_value' in out.columns:
        out['fg_value'] = out['Feer_Greed_value'].shift(1)
        out['fg_chg'] = out['Feer_Greed_value'].diff().shift(1)
    # Контемпоранные BTC-производные содержат BTC_t -> сдвигаем на 1 бар.
    present = [c for c in CONTEMP_BTC_COLS if c in out.columns]
    out[present] = out[present].shift(1)
    needed = list(dict.fromkeys(list(feature_cols) + ['BTC', 'BTC_logret']))
    return out.dropna(subset=needed)


wf_frames = OrderedDict(
    (freq, prep_returns(datasets[freq], TOP_FEATURES[freq])) for freq in DATA_CONFIG
)

# Сводка по подготовленным фреймам и пулу признаков-кандидатов.
split_summary = pd.DataFrame([
    {
        'frequency': freq,
        'rows': len(frame),
        'start': frame.index.min(),
        'end': frame.index.max(),
        'logret_mean': frame['BTC_logret'].mean(),
        'logret_std': frame['BTC_logret'].std(),
        'candidate_features': len(TOP_FEATURES[freq]),
    }
    for freq, frame in wf_frames.items()
])

feature_table = pd.DataFrame([
    {'frequency': freq, 'rank': rank, 'feature': feat}
    for freq in wf_frames
    for rank, feat in enumerate(TOP_FEATURES[freq], start=1)
])

display(split_summary)
display(feature_table)


## Experiment Setup and Results

All models are evaluated on the same time-based splits per frequency. Gradient Boosting and baseline models use a fixed train-only feature shortlist (`TOP_FEATURES`) and hardcoded hyperparameters (`GB_HYPERPARAMS`). Each of the four NN architectures (LSTM, GRU, Stacked LSTM, CNN-LSTM) uses a windowed sequence representation (`NN_CONFIG`), with hyperparameters chosen per architecture/frequency by a small Optuna search (`NN_OPTUNA_TRIALS` trials), and train-only `RobustScaler` fits for both the BTC sequence and the exogenous features. Baselines do not use future observations. All scalers, imputers, and clipping bounds are fit on training data only.


In [ ]:
def build_model_specs():
    """Gradient boosting candidates only, with hardcoded hyperparameters."""
    specs = OrderedDict()

    specs['HistGradientBoosting'] = make_preprocessed_model(
        HistGradientBoostingRegressor(**GB_HYPERPARAMS['HistGradientBoosting']),
        scale=False,
    )

    if HAS_LIGHTGBM:
        specs['LightGBM'] = make_preprocessed_model(
            LGBMRegressor(**GB_HYPERPARAMS['LightGBM']),
            scale=False,
        )

    if HAS_XGBOOST:
        specs['XGBoost'] = make_preprocessed_model(
            XGBRegressor(**GB_HYPERPARAMS['XGBoost']),
            scale=False,
        )

    if HAS_CATBOOST:
        specs['CatBoost'] = make_preprocessed_model(
            CatBoostRegressor(**GB_HYPERPARAMS['CatBoost']),
            scale=False,
        )

    return specs


In [ ]:
import time as _time
from sklearn.base import clone

from walk_forward import (
    expanding_window_splits, rolling_window_splits, run_walk_forward,
    select_best_by_composite,
)
from backtest import simulate_strategy, buy_and_hold, backtest_summary, round_trip_cost
from ml_models import compute_metrics
from metrics import rank_ic, ic_ir, mwda, mcc, balanced_accuracy, net_cost_sharpe

TARGET = 'BTC_logret'


def _family(name):
    if 'sign clf' in name:
        return 'NN_CLF'
    if any(a in name for a in ('LSTM', 'GRU', 'CNN_LSTM', 'StackedLSTM')):
        return 'NN'
    if name in ('Naive (zero return)', 'Last return', 'Moving average',
                'Always long', 'Base rate'):
        return 'baseline'
    return 'GB'


def _get_splits(frequency, cfg, n):
    if cfg['type'] == 'expanding':
        return expanding_window_splits(n, cfg['n_splits'], cfg['min_train_frac'])
    return rolling_window_splits(n, cfg['train_size'], cfg['test_size'], n_splits=cfg['n_splits'])


def make_gb_factory(model_template):
    """Zero-arg callable producing a fresh clone of model_template."""
    def factory():
        return clone(model_template)
    return factory


def reconstruct_price_oof(oof_ret_df, price):
    """Восстановление цены из прогноза доходности: P_hat_t = P_{t-1} * exp(r_hat_t).

    oof_ret_df - OOF в пространстве доходностей (actual/predicted = лог-доходности).
    price      - полный ряд уровня цены BTC (datasets[freq]['BTC']).
    Возвращает OOF в пространстве цены для экономического бэктеста.
    """
    prev_price = price.shift(1)
    rows = []
    for model_name, grp in oof_ret_df.groupby('model'):
        grp = grp.sort_values('timestamp')
        p_prev = prev_price.reindex(grp['timestamp']).to_numpy(dtype=float)
        p_actual = price.reindex(grp['timestamp']).to_numpy(dtype=float)
        # клип прогноза доходности перед exp: защита от переполнения, если модель
        # разойдётся на коротком фолде (|r_hat| > 1 экономически нереалистично).
        r_hat = np.clip(grp['predicted'].to_numpy(dtype=float), -1.0, 1.0)
        p_pred = p_prev * np.exp(r_hat)
        rows.append(pd.DataFrame({
            'timestamp': grp['timestamp'].values,
            'actual': p_actual,
            'predicted': p_pred,
            'fold': grp['fold'].values,
            'model': model_name,
        }))
    return pd.concat(rows, ignore_index=True)


wf_metrics_all = {}      # freq -> per-fold metrics (return space)
wf_oof_all = {}          # freq -> OOF predictions (return space)
wf_oof_price_all = {}    # freq -> OOF reconstructed to price (for the backtest)

t_wf_start = _time.time()

for frequency, df_freq in wf_frames.items():
    print(f'\n=== Walk-forward: {frequency} (target = log-return) ===')

    n = len(df_freq)
    cfg = SPLIT_CONFIG[frequency]
    splits = _get_splits(frequency, cfg, n)
    nn_cfg = NN_CONFIG[frequency]
    feat_pool = TOP_FEATURES[frequency]

    freq_metrics_list = []
    freq_oof_list = []

    # 1. Gradient Boosting models
    for gb_name, gb_model_tpl in build_model_specs().items():
        print(f'  GB {gb_name}...', end=' ', flush=True)
        t0 = _time.time()
        mdf, odf = run_walk_forward(
            model_family='GB', model_name=gb_name,
            df=df_freq, target_col=TARGET, all_feature_cols=feat_pool,
            splits=splits, top_k=min(WF_TOP_K, len(feat_pool)),
            optuna_n_trials=WF_OPTUNA_TRIALS,
            gb_model_factory=make_gb_factory(gb_model_tpl),
            seed=RANDOM_STATE,
            retune_every=RETUNE_EVERY,
        )
        print(f'{_time.time()-t0:.0f}s  DA={mdf["DA"].mean():.3f}  MASE={mdf["MASE"].mean():.3f}')
        freq_metrics_list.append(mdf)
        freq_oof_list.append(odf)

    # 2. NN architectures
    for arch in NN_ARCHITECTURES:
        model_name = f'{arch} (sequence NN)'
        print(f'  NN {arch}...', end=' ', flush=True)
        t0 = _time.time()
        mdf, odf = run_walk_forward(
            model_family='NN', model_name=model_name,
            df=df_freq, target_col=TARGET, all_feature_cols=feat_pool,
            splits=splits, top_k=min(WF_TOP_K, len(feat_pool)),
            optuna_n_trials=WF_NN_OPTUNA_TRIALS,
            nn_config=nn_cfg,
            seed=RANDOM_STATE,
            retune_every=RETUNE_EVERY,
        )
        print(f'{_time.time()-t0:.0f}s  DA={mdf["DA"].mean():.3f}  MASE={mdf["MASE"].mean():.3f}')
        freq_metrics_list.append(mdf)
        freq_oof_list.append(odf)

    # 2b. NN sign classifiers (weighted BCE on direction)
    if RUN_NN_CLF:
        for arch in NN_ARCHITECTURES:
            model_name = f'{arch} (sign clf)'
            print(f'  CLF {arch}...', end=' ', flush=True)
            t0 = _time.time()
            mdf, odf = run_walk_forward(
                model_family='NN_CLF', model_name=model_name,
                df=df_freq, target_col=TARGET, all_feature_cols=feat_pool,
                splits=splits, top_k=min(WF_TOP_K, len(feat_pool)),
                optuna_n_trials=WF_NN_OPTUNA_TRIALS,
                nn_config=nn_cfg,
                seed=RANDOM_STATE,
                retune_every=RETUNE_EVERY,
            )
            print(f'{_time.time()-t0:.0f}s  DA={mdf["DA"].mean():.3f}  AUC={mdf["AUC"].mean():.3f}')
            freq_metrics_list.append(mdf)
            freq_oof_list.append(odf)

    # 3. Baselines in return space:
    #    Naive (zero return) -> r_hat = 0   (бенчмарк "случайное блуждание цены")
    #    Last return         -> r_hat_t = r_{t-1}   (персистентность доходности)
    #    Moving average      -> скользящее среднее прошлых доходностей
    #    Always long         -> r_hat = +eps   (всегда длинная позиция; DA = base-rate up)
    #    Base rate           -> знак мажоритарного класса доходности на трейне (T3.4)
    ret = df_freq[TARGET]
    ma_window = DATA_CONFIG[frequency]['ma_window']
    EPS_DIR = 1e-6
    for fold_idx, (train_idx, test_idx) in enumerate(splits):
        actual = ret.iloc[test_idx]
        y_tr = ret.iloc[train_idx].to_numpy(dtype=float)
        base_up = float(np.mean(y_tr > 0)) if y_tr.size else 0.5
        base_sign = EPS_DIR if base_up >= 0.5 else -EPS_DIR
        preds = {
            'Naive (zero return)': pd.Series(0.0, index=actual.index),
            'Last return': ret.shift(1).iloc[test_idx],
            'Moving average': ret.shift(1).rolling(ma_window, min_periods=1).mean().iloc[test_idx],
            'Always long': pd.Series(EPS_DIR, index=actual.index),
            'Base rate': pd.Series(base_sign, index=actual.index),
        }
        for bname, bpred in preds.items():
            ba = actual.to_numpy(dtype=float)
            bp = bpred.to_numpy(dtype=float)
            m = compute_metrics(ba, bp, y_tr)
            is_dir = bname in ('Always long', 'Base rate')
            freq_metrics_list.append(pd.DataFrame([{
                'fold': fold_idx, 'model': bname,
                'MAE': m['mae'], 'RMSE': m['rmse'], 'SMAPE': m['smape'],
                'MASE': m['mase'], 'DA': m['da'],
                'RankIC': rank_ic(ba, bp), 'MWDA': mwda(ba, bp),
                'MCC': mcc(ba, bp) if is_dir else np.nan,
                'BalAcc': balanced_accuracy(ba, bp) if is_dir else np.nan,
                'n_train': len(train_idx), 'n_test': len(test_idx),
            }]))
            freq_oof_list.append(pd.DataFrame({
                'timestamp': actual.index, 'actual': ba,
                'predicted': bp, 'fold': fold_idx, 'model': bname,
            }))

    wf_metrics_all[frequency] = pd.concat(freq_metrics_list, ignore_index=True)
    wf_oof_all[frequency] = pd.concat(freq_oof_list, ignore_index=True)
    wf_oof_price_all[frequency] = reconstruct_price_oof(
        wf_oof_all[frequency], datasets[frequency]['BTC']
    )

print(f'\nTotal walk-forward time: {_time.time()-t_wf_start:.0f}s')

# Aggregate (mean across folds) + composite-selection metrics (EPIC 3, T3.1).
# Ведущая метрика селекции — IC IR (стабильность Rank IC по фолдам), затем
# net-of-cost Sharpe, затем MWDA; tie-break MASE. (Раньше — «голая» DA.)
RTC_FEE = round_trip_cost(0.0005, 2.0)   # ~0.14% round-trip для net-of-cost Sharpe
agg_frames = []
for freq, mdf in wf_metrics_all.items():
    metric_cols = [c for c in ['MAE', 'RMSE', 'MASE', 'DA', 'RankIC', 'MWDA', 'MCC', 'BalAcc', 'AUC']
                   if c in mdf.columns]
    agg = mdf.groupby('model')[metric_cols].mean().reset_index()
    # IC IR = mean(IC)/std(IC) по фолдам (стабильность), НЕ среднее IC.
    ic_ir_by_model = mdf.groupby('model')['RankIC'].apply(lambda s: ic_ir(s.dropna().tolist()))
    agg['IC_IR'] = agg['model'].map(ic_ir_by_model)
    # net-of-cost Sharpe в return-space: side = sign(r_hat), издержки на оборот позиции.
    oof = wf_oof_all[freq]
    ncs = {}
    for mname, grp in oof.groupby('model'):
        g = grp.sort_values('timestamp')
        ncs[mname] = net_cost_sharpe(np.sign(g['predicted'].to_numpy(dtype=float)),
                                     g['actual'].to_numpy(dtype=float), fee=RTC_FEE)
    agg['NetSharpe'] = agg['model'].map(ncs)
    agg['frequency'] = freq
    agg['family'] = agg['model'].apply(_family)
    agg_frames.append(agg)
wf_agg = pd.concat(agg_frames, ignore_index=True)

_gb_names = set(build_model_specs().keys())
_nn_names = {f'{a} (sequence NN)' for a in NN_ARCHITECTURES}
_clf_names = {f'{a} (sign clf)' for a in NN_ARCHITECTURES}


def _best_by_composite(names):
    # Делегирует в импортируемую walk_forward.select_best_by_composite (тестируемо):
    # IC IR -> net-of-cost Sharpe -> MWDA, tie-break MASE asc (EPIC 3, T3.1).
    return select_best_by_composite(wf_agg, names)


best_gb = _best_by_composite(_gb_names)
best_nn = _best_by_composite(_nn_names)
best_nn_clf = _best_by_composite(_clf_names)   # пусто, если RUN_NN_CLF=False

# Price-space continuity metrics (SMAPE/MAE on reconstructed price) for comparability
# with the prior level-target report. SMAPE is well-defined here (prices > 0).
price_metric_rows = []
for freq, pdf in wf_oof_price_all.items():
    for model_name, grp in pdf.groupby('model'):
        if _family(model_name) == 'NN_CLF':
            continue  # классификатор не предсказывает уровень цены -> SMAPE_price не определён
        g = grp.dropna(subset=['actual', 'predicted'])
        if g.empty:
            continue
        price_metric_rows.append({
            'frequency': freq, 'model': model_name, 'family': _family(model_name),
            'SMAPE_price': 100 * smape(g['actual'].values, g['predicted'].values),
            'MAE_price': mean_absolute_error(g['actual'], g['predicted']),
        })
price_metrics = pd.DataFrame(price_metric_rows)

# Refit best GB / NN on the LAST fold's train data, on the SAME shifted return frame
# used by walk-forward (no raw-price leakage) -> interpretation matches evaluation.
refit_gb = {}
for _, row in best_gb.iterrows():
    freq, mname = row['frequency'], row['model']
    frame = wf_frames[freq]
    last_train_idx = _get_splits(freq, SPLIT_CONFIG[freq], len(frame))[-1][0]
    X_last = frame[TOP_FEATURES[freq]].iloc[last_train_idx]
    y_last = frame[TARGET].iloc[last_train_idx]
    m = clone(build_model_specs()[mname])
    m.fit(X_last, y_last)
    refit_gb.setdefault(freq, {})[mname] = m

refit_nn = {}
hp_map = {
    'LSTM': {'lstm_units': 64, 'lstm_drop': 0.2, 'ex_layers': 1, 'ex_units_0': 32, 'ex_drop_0': 0.2},
    'GRU': {'gru_units': 64, 'gru_drop': 0.2, 'ex_layers': 1, 'ex_units_0': 32, 'ex_drop_0': 0.2},
    'StackedLSTM': {'stacked_layers': 2, 'stack_units_0': 64, 'stack_drop_0': 0.2, 'stack_units_1': 64, 'stack_drop_1': 0.2, 'ex_layers': 1, 'ex_units_0': 32, 'ex_drop_0': 0.2},
    'CNN_LSTM': {'cnn_kernel': 3, 'pool_size': 2, 'cnn_filters': 32, 'cnn_drop': 0.2, 'cnn_lstm_units': 64, 'ex_layers': 1, 'ex_units_0': 32, 'ex_drop_0': 0.2},
}
for _, row in best_nn.iterrows():
    freq, mname = row['frequency'], row['model']
    arch = mname.split()[0]
    nn_cfg = NN_CONFIG[freq]
    frame = wf_frames[freq]
    last_train_idx = _get_splits(freq, SPLIT_CONFIG[freq], len(frame))[-1][0]
    seq_cols = TOP_FEATURES[freq] + [TARGET]
    seq_train_df = frame[seq_cols].iloc[last_train_idx]
    reg = CryptoNetRegressor(
        arch=arch, target_col=TARGET, feature_cols=TOP_FEATURES[freq],
        window_size=nn_cfg['window_size'], hp=hp_map[arch],
        lr=1e-3, epochs=nn_cfg['epochs'], batch_size=nn_cfg['batch_size'],
        patience=5, seed=RANDOM_STATE,
    )
    reg.fit(seq_train_df)
    refit_nn.setdefault(freq, {})[mname] = reg

# prediction_store: OOF predictions (RETURN space) + fitted model for interpretation cells.
prediction_store = OrderedDict()
for freq, oof_df in wf_oof_all.items():
    prediction_store[freq] = {}
    for model_name, grp in oof_df.groupby('model'):
        grp = grp.sort_values('timestamp')
        fitted_obj = refit_gb.get(freq, {}).get(model_name) or refit_nn.get(freq, {}).get(model_name)
        prediction_store[freq][model_name] = {
            'test_pred': pd.Series(grp['predicted'].values, index=grp['timestamp']),
            'train_pred': pd.Series(dtype=float),
            'model': fitted_obj,
        }

results_df = wf_agg.copy()
results_df['comment'] = 'Walk-forward OOF mean across folds (log-return target)'


In [ ]:
# Per-fold metrics (return space). SMAPE опущена: на доходностях знаменатель
# обращается в ноль при смене знака; ведущая метрика — Directional Accuracy (DA).
# Непрерывность с прежним отчётом даёт отдельная таблица метрик на восстановленной
# ЦЕНЕ ниже.
print('Walk-forward per-fold metrics (target = log-return):')
for freq in wf_metrics_all:
    print(f'\n--- {freq} ---')
    fold_table = (wf_metrics_all[freq]
                  .sort_values(['model', 'fold'])
                  [['fold', 'model', 'MAE', 'RMSE', 'MASE', 'DA', 'AUC', 'n_train', 'n_test']])
    display(fold_table.round({'MAE': 6, 'RMSE': 6, 'MASE': 4, 'DA': 4, 'AUC': 4}))

print('\n\nWalk-forward aggregate (mean across folds, sorted by IC IR — composite lead):')
agg_display = wf_agg.sort_values(['frequency', 'IC_IR'], ascending=[True, False]).copy()
_agg_cols = ['frequency', 'model', 'family', 'MASE', 'DA', 'IC_IR', 'RankIC',
             'MWDA', 'NetSharpe', 'MCC', 'BalAcc', 'AUC']
display(agg_display[[c for c in _agg_cols if c in agg_display.columns]]
        .round({'MASE': 4, 'DA': 4, 'IC_IR': 3, 'RankIC': 4, 'MWDA': 4,
                'NetSharpe': 3, 'MCC': 4, 'BalAcc': 4, 'AUC': 4}))

# Headline: Directional Accuracy of naive + best GB + best NN per frequency.
# Опорная линия — max(base_rate, 1−base_rate) (always-majority baseline), а НЕ 0.5:
# при дрейфе BTC доля up-баров != 0.5, и честный нуль навыка = доля мажоритарного класса.
base_rate_by_freq = {}
for _freq in wf_frames:
    _r = wf_frames[_freq][TARGET].to_numpy(dtype=float)
    _r = _r[np.isfinite(_r)]
    _up = float(np.mean(_r > 0)) if _r.size else 0.5
    base_rate_by_freq[_freq] = max(_up, 1.0 - _up)
baseline_agg = wf_agg[wf_agg['family'] == 'baseline']
headline = (pd.concat([baseline_agg, best_gb, best_nn, best_nn_clf])
            .sort_values(['frequency', 'DA'], ascending=[True, False]).reset_index(drop=True))

fig, axes = plt.subplots(1, len(wf_frames), figsize=(6 * len(wf_frames), 5), sharey=False)
if len(wf_frames) == 1:
    axes = [axes]
for ax, frequency in zip(axes, wf_frames.keys()):
    subset = headline[headline['frequency'] == frequency].sort_values('DA')
    sns.barplot(data=subset, x='DA', y='model', hue='model', legend=False, ax=ax, palette='viridis')
    _naive_da = base_rate_by_freq[frequency]
    ax.axvline(_naive_da, color='red', linestyle='--', linewidth=1)
    ax.set_title(f'{frequency.title()} Directional Accuracy (walk-forward OOF)')
    ax.set_xlabel(f'DA (mean across folds); {_naive_da:.3f} = always-majority baseline')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

print('Best model per family by composite (IC IR -> NetSharpe -> MWDA; incl. sign clf):')
_hl_cols = ['frequency', 'model', 'family', 'MASE', 'DA', 'IC_IR', 'MWDA',
            'NetSharpe', 'MCC', 'BalAcc', 'AUC']
display(headline[[c for c in _hl_cols if c in headline.columns]]
        .round({'MASE': 4, 'DA': 4, 'IC_IR': 3, 'MWDA': 4, 'NetSharpe': 3,
                'MCC': 4, 'BalAcc': 4, 'AUC': 4}))

# Price-space continuity metrics (comparable to the prior level-target SMAPE).
print('\n\nReconstructed-price error (continuity with prior report):')
display(price_metrics.sort_values(['frequency', 'SMAPE_price'])
        .round({'SMAPE_price': 4, 'MAE_price': 2}))

# -------------------------------------------------------
# Экономический бэктест: naive (порог 0) vs cost-aware (порог + гистерезис + min-hold)
# -------------------------------------------------------
from backtest import round_trip_cost, threshold_sweep, select_threshold_by_sharpe

RTC = round_trip_cost(0.0005, 2.0)        # полные издержки входа+выхода (~0.14%)
MIN_HOLD = {'daily': 2, 'hourly': 4, '5min': 6}
print(f'\n\nRound-trip cost = {RTC:.4%}. Cost-aware: вход при r_hat > RTC, выход при '
      f'r_hat < 0 (гистерезис), min-hold = {MIN_HOLD}.')

econ_rows = []
for freq, price_oof in wf_oof_price_all.items():
    ppy = PERIODS_PER_YEAR[freq]
    bh = buy_and_hold(price_oof)
    variants = {
        'thr=0 (как раньше)': dict(enter_threshold=0.0, exit_threshold=0.0, min_hold=1),
        'cost-aware':         dict(enter_threshold=RTC, exit_threshold=0.0, min_hold=MIN_HOLD[freq]),
    }
    for label, kw in variants.items():
        strat = simulate_strategy(price_oof, fee_rate=0.0005, slippage_bps=2.0, **kw)
        summ = backtest_summary(strat, bh, periods_per_year=ppy)
        st = summ[summ['type'] == 'Strategy'].copy()
        st['variant'] = label; st['frequency'] = freq
        econ_rows.append(st)
        if label == 'cost-aware':
            bo = summ[summ['type'] == 'Buy&Hold'].copy()
            bo['variant'] = 'Buy&Hold'; bo['frequency'] = freq
            econ_rows.append(bo)
econ_df = pd.concat(econ_rows, ignore_index=True)

for freq in wf_frames:
    print(f'\n--- {freq} ---')
    subset = econ_df[econ_df['frequency'] == freq].sort_values(['model', 'variant'])
    display(subset[['model', 'variant', 'total_return_usd', 'total_fees_usd',
                    'sharpe', 'max_drawdown_usd', 'turnover']]
            .round({'total_return_usd': 2, 'total_fees_usd': 2, 'sharpe': 3, 'max_drawdown_usd': 2}))

# Кривая «порог -> доход / оборот» для лучшей по DA модели каждой частоты:
# показывает, как растущий порог режет оборот (и комиссии) и где оптимум по доходу.
print('\nThreshold sweep (порог в единицах round-trip cost): net return и turnover')
_focus = (headline[headline['family'] != 'baseline']
          .sort_values('DA', ascending=False).groupby('frequency').head(1))
fig, axes = plt.subplots(1, len(wf_frames), figsize=(6 * len(wf_frames), 4))
if len(wf_frames) == 1:
    axes = [axes]
grid = np.array([0, 0.5, 1, 1.5, 2, 3, 5]) * RTC
for ax, freq in zip(axes, wf_frames.keys()):
    bm = _focus[_focus['frequency'] == freq]['model']
    if bm.empty:
        continue
    bm = bm.iloc[0]
    po = wf_oof_price_all[freq]
    po = po[po['model'] == bm]
    sw = threshold_sweep(po, thresholds=grid, periods_per_year=PERIODS_PER_YEAR[freq],
                         min_hold=MIN_HOLD[freq])
    ax2 = ax.twinx()
    ax.plot(sw['threshold'] / RTC, sw['total_return_usd'], 'o-', color='tab:green')
    ax2.plot(sw['threshold'] / RTC, sw['turnover'], 's--', color='tab:red')
    ax.axhline(0, color='grey', lw=0.6)
    ax.set_title(f'{freq}: {bm}')
    ax.set_xlabel('порог / round-trip cost')
    ax.set_ylabel('net return $', color='tab:green')
    ax2.set_ylabel('turnover', color='tab:red')
plt.tight_layout(); plt.show()

# Порог, подобранный по net-of-cost Sharpe на ВАЛИДАЦИИ (первые 40% OOF), оценённый на
# оставшейся тест-части — без подглядывания в тестовый период.
print('\nVal-tuned threshold (выбран на первых 40% OOF, оценён на остатке):')
grid_sel = np.array([0, 0.5, 1, 1.5, 2, 3, 5, 8]) * RTC
tuned_rows = []
for freq, price_oof in wf_oof_price_all.items():
    ppy = PERIODS_PER_YEAR[freq]
    focus_models = _focus[_focus['frequency'] == freq]['model'].tolist()
    sel = select_threshold_by_sharpe(price_oof, grid_sel, periods_per_year=ppy,
                                     val_frac=0.4, min_hold=MIN_HOLD[freq])
    for model in focus_models:
        g = price_oof[price_oof['model'] == model].sort_values('timestamp')
        n_val = max(20, int(len(g) * 0.4))
        test = g.iloc[n_val:]
        thr = sel.get(model, RTC)
        strat = simulate_strategy(test, enter_threshold=thr, exit_threshold=0.0, min_hold=MIN_HOLD[freq])
        summ = backtest_summary(strat, buy_and_hold(test), periods_per_year=ppy)
        stt = summ[summ['type'] == 'Strategy'].iloc[0]
        bht = summ[summ['type'] == 'Buy&Hold'].iloc[0]
        tuned_rows.append({'frequency': freq, 'model': model, 'thr/cost': round(thr / RTC, 1),
                           'strat_return': stt['total_return_usd'], 'strat_sharpe': stt['sharpe'],
                           'turnover': stt['turnover'], 'BH_return': bht['total_return_usd'],
                           'BH_sharpe': bht['sharpe']})
display(pd.DataFrame(tuned_rows).round({'strat_return': 2, 'strat_sharpe': 3,
                                        'BH_return': 2, 'BH_sharpe': 3}))


## EPIC 4 — двухэтапный мета-лейблинг

Stage 1 (sequence-NN) задаёт **сторону** `sign(r̂)`; Stage 2 (GBM) — **уверенность**
`p_ok = P(ставка верна)` для сайзинга и фильтрации сделок (López de Prado, AFML гл. 3).
Stage 2 учится СТРОГО на OOF первичной модели собственным каузальным walk-forward
(purge/embargo = горизонт). Перед внедрением — sanity-gate: OOS Rank IC регрессора
остатка `e = r − r̂`; если ≈ 0, структурного остатка нет. Ниже — сравнение Stage 1
(только сторона) против Stage 1+2 (фильтр+сайзинг) в return-space (net-of-cost Sharpe,
precision, turnover). Непрерывный `size` уходит в бэктест в EPIC 5.

In [ ]:
# =====================================================================
# EPIC 4 — двухэтапный мета-лейблинг (Stage 1 = NN сторона, Stage 2 = GBM уверенность)
# =====================================================================
from meta_labeling import (build_meta_dataset, train_meta_walkforward,
                           meta_size, residual_sanity_gate)
from metrics import net_cost_sharpe

META_FEE = round_trip_cost(0.0005, 2.0)   # round-trip издержки для net-of-cost Sharpe

meta_rows = []
meta_oof = {}   # freq -> DataFrame(timestamp, side, p_ok, size, actual) для EPIC 5
for freq in wf_frames:
    # Stage 1 = лучшая по композиту sequence-NN (гипотеза диплома); fallback — лучший GB.
    primary = None
    if not best_nn.empty and (best_nn['frequency'] == freq).any():
        primary = best_nn[best_nn['frequency'] == freq]['model'].iloc[0]
    elif not best_gb.empty and (best_gb['frequency'] == freq).any():
        primary = best_gb[best_gb['frequency'] == freq]['model'].iloc[0]
    if primary is None:
        continue

    oof = wf_oof_all[freq]
    oof = oof[oof['model'] == primary][['timestamp', 'actual', 'predicted']].dropna()
    if oof.empty:
        continue
    feats = wf_frames[freq][TOP_FEATURES[freq]]                 # лагированные exog-фичи
    horizon = DATA_CONFIG[freq].get('horizon', 1)

    # sanity-gate (T4.5): есть ли вообще структурный остаток
    ic_res = residual_sanity_gate(oof, feats, horizon=horizon)
    if not np.isfinite(ic_res) or abs(ic_res) < 0.01:
        print(f'[{freq}] WARNING: residual Rank IC={ic_res:.3f} ~ 0 — структурного '
              f'остатка нет; мета-этап может не дать сигнала.')

    md = build_meta_dataset(oof, feats, include_pred=True)
    p_ok = train_meta_walkforward(md, horizon=horizon, n_splits=8, min_train_frac=0.3)
    size = meta_size(md.side, p_ok)
    side = md.side
    r = oof.sort_values('timestamp')['actual'].to_numpy(dtype=float)
    ppy = PERIODS_PER_YEAR[freq]

    traded = size != 0
    base_turn = int(np.sum(np.abs(np.diff(np.concatenate([[0.0], side]))) > 1e-9))
    meta_turn = int(np.sum(np.abs(np.diff(np.concatenate([[0.0], size]))) > 1e-9))
    meta_rows.append({
        'frequency': freq, 'primary (Stage 1)': primary,
        'resid_rank_ic': ic_res, 'meta_coverage': float(np.isfinite(p_ok).mean()),
        'sharpe_side': net_cost_sharpe(side, r, fee=META_FEE, ann=ppy),
        'sharpe_meta': net_cost_sharpe(size, r, fee=META_FEE, ann=ppy),
        'precision_side': float(np.mean(md.y)) if md.y.size else np.nan,
        'precision_meta': float(np.mean(md.y[traded])) if traded.any() else np.nan,
        'turnover_side': base_turn, 'turnover_meta': meta_turn,
        'trade_frac_meta': float(np.mean(traded)),
    })
    meta_oof[freq] = pd.DataFrame({'timestamp': md.timestamp, 'side': side,
                                   'p_ok': p_ok, 'size': size, 'actual': r})

meta_summary = pd.DataFrame(meta_rows)
print('Meta-labeling: Stage 1 (side only) vs Stage 1+2 (confidence filter & sizing)')
print('Гипотеза: precision_meta > precision_side и turnover_meta < turnover_side '
      '(отсечение ложных сигналов), net-of-cost Sharpe растёт.')
display(meta_summary.round({'resid_rank_ic': 3, 'meta_coverage': 2, 'sharpe_side': 3,
        'sharpe_meta': 3, 'precision_side': 3, 'precision_meta': 3, 'trade_frac_meta': 2}))

## Interpretation

Feature importance is computed for the best Gradient Boosting model per frequency using permutation importance (model-agnostic, test segment) and SHAP (`TreeExplainer`, when available). For the GRU, feature importance is approximated with gradient-based attribution (mean absolute gradient of the prediction with respect to each scaled exogenous feature), since SHAP's tree explainer does not apply to a PyTorch model. Comparing the two rankings is the basis for research question 3: do GB and the GRU rely on similar predictive signals?


In [ ]:
def permutation_importance_table(frequency, model_name, n_repeats=5, top_n=15):
    model = prediction_store[frequency][model_name]['model']
    if model is None or not hasattr(model, 'predict'):
        return pd.DataFrame()
    # Use last walk-forward fold's test data for permutation importance
    n = len(wf_frames[frequency])
    last_train_idx, last_test_idx = _get_splits(frequency, SPLIT_CONFIG[frequency], n)[-1]
    X_test_last = wf_frames[frequency][TOP_FEATURES[frequency]].iloc[last_test_idx]
    y_test_last = wf_frames[frequency]['BTC_logret'].iloc[last_test_idx]
    importance = permutation_importance(
        model,
        X_test_last,
        y_test_last,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
        scoring='neg_mean_absolute_error',
        n_jobs=1,
    )
    table = pd.DataFrame({
        'frequency': frequency,
        'model': model_name,
        'feature': X_test_last.columns,
        'importance_mean': importance.importances_mean,
        'importance_std': importance.importances_std,
    }).sort_values('importance_mean', ascending=False).head(top_n)
    return table

importance_tables = []
for _, row in best_gb.iterrows():
    table = permutation_importance_table(row['frequency'], row['model'])
    if not table.empty:
        importance_tables.append(table)

if importance_tables:
    importance_df = pd.concat(importance_tables, ignore_index=True)
    display(importance_df.round(4))

    fig, axes = plt.subplots(len(importance_tables), 1, figsize=(14, 4 * len(importance_tables)))
    if len(importance_tables) == 1:
        axes = [axes]
    for ax, table in zip(axes, importance_tables):
        sns.barplot(data=table, x='importance_mean', y='feature', ax=ax, color='tab:blue')
        frequency = table['frequency'].iloc[0]
        model_name = table['model'].iloc[0]
        ax.set_title(f'{frequency.title()} permutation importance: {model_name} (last WF fold)')
        ax.set_xlabel('Increase in MAE after permutation')
        ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print('No permutation importance table was produced for the selected GB models.')


In [ ]:
# SHAP interpretation for the best Gradient Boosting model per frequency.
# Uses the refit model (trained on last WF fold's train data) and that fold's test data.
try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

if HAS_SHAP:
    for _, row in best_gb.iterrows():
        frequency = row['frequency']
        model_name = row['model']
        model = prediction_store[frequency][model_name]['model']
        if model is None:
            print(f'No refit model for {frequency} {model_name}, skipping SHAP.')
            continue
        # Last WF fold test data (same as permutation importance)
        n = len(wf_frames[frequency])
        _, last_test_idx = _get_splits(frequency, SPLIT_CONFIG[frequency], n)[-1]
        X_test_last = wf_frames[frequency][TOP_FEATURES[frequency]].iloc[last_test_idx]
        sample = X_test_last.sample(min(500, len(X_test_last)), random_state=RANDOM_STATE)
        estimator = model.named_steps['model'] if isinstance(model, Pipeline) else model
        transformed = model[:-1].transform(sample) if isinstance(model, Pipeline) else sample
        try:
            explainer = shap.TreeExplainer(estimator)
            shap_values = explainer.shap_values(transformed)
            shap.summary_plot(shap_values, sample, plot_type='bar', show=True)
        except Exception as exc:
            print(f'SHAP failed for {frequency} {model_name}: {exc}')
else:
    print('SHAP is not installed in the active environment; permutation importance above is used as the reproducible interpretation method for the GB models.')


In [ ]:
# Captum Integrated Gradients attribution for the best NN per frequency.
# Attributes the exogenous-feature branch of CryptoNet; the BTC-sequence branch
# is held fixed (its values are reused at every interpolation step), isolating
# the marginal contribution of each exogenous feature relative to a zero baseline.
try:
    from captum.attr import IntegratedGradients
    HAS_CAPTUM = True
except ImportError:
    HAS_CAPTUM = False

import torch

nn_attribution_tables = []
for _, row in best_nn.iterrows():
    frequency = row['frequency']
    model_name = row['model']
    reg = prediction_store[frequency][model_name]['model']
    if reg is None or not hasattr(reg, 'model_'):
        print(f'No fitted NN model available for {frequency} {model_name}, skipping.')
        continue

    # Last WF fold test data with window_size history prefix
    n = len(wf_frames[frequency])
    last_train_idx, last_test_idx = _get_splits(frequency, SPLIT_CONFIG[frequency], n)[-1]
    hist_start = max(0, int(last_train_idx[-1]) + 1 - reg.window_size)
    seq_test_df = wf_frames[frequency][TOP_FEATURES[frequency] + ['BTC_logret']].iloc[
        hist_start : int(last_test_idx[-1]) + 1
    ]

    X_seq, X_ex, _, _ = reg._make_windows(seq_test_df)
    X_seq_s = reg.scaler_y_.transform(X_seq.reshape(-1, 1)).reshape(X_seq.shape)
    X_ex_s = reg.scaler_x_.transform(X_ex)

    x_seq_t = torch.tensor(X_seq_s, dtype=torch.float32, device=DEVICE)
    x_ex_t = torch.tensor(X_ex_s, dtype=torch.float32, device=DEVICE)

    reg.model_.eval()

    if HAS_CAPTUM:
        # Captum IG атрибутирует ТОЛЬКО экзогенную ветвь; ветвь BTC-последовательности
        # фиксируется и передаётся как additional_forward_args, чтобы Captum размножил её
        # по n_steps интерполяции синхронно с x_ex. Замыкание над x_seq_t давало
        # рассинхрон батча (x_seq оставался B, x_ex расширялся до B*n_steps -> ошибка cat).
        def _ig_forward(x_ex, x_seq):
            return reg.model_(x_seq, x_ex).squeeze(-1)   # (N,1) -> (N,), чтобы IG не требовал target
        ig = IntegratedGradients(_ig_forward)
        attributions, convergence_delta = ig.attribute(
            x_ex_t,
            baselines=torch.zeros_like(x_ex_t),
            additional_forward_args=(x_seq_t,),
            return_convergence_delta=True,
        )
        importance = attributions.detach().abs().cpu().numpy().mean(axis=0)
        method_label = 'Captum IntegratedGradients'
    else:
        # Fallback to plain gradient attribution when captum is unavailable
        x_ex_grad = x_ex_t.clone().requires_grad_(True)
        reg.model_(x_seq_t, x_ex_grad).sum().backward()
        importance = x_ex_grad.grad.detach().abs().cpu().numpy().mean(axis=0)
        method_label = 'gradient attribution (backward)'

    feature_names = reg.feature_cols if hasattr(reg, 'feature_cols') and reg.feature_cols else TOP_FEATURES[frequency]
    table = pd.DataFrame({
        'frequency': frequency,
        'model': model_name,
        'feature': feature_names,
        'importance_mean': importance,
        'method': method_label,
    }).sort_values('importance_mean', ascending=False).head(15)
    nn_attribution_tables.append(table)

if nn_attribution_tables:
    nn_attribution_df = pd.concat(nn_attribution_tables, ignore_index=True)
    display(nn_attribution_df.round(4))

    fig, axes = plt.subplots(len(nn_attribution_tables), 1, figsize=(14, 4 * len(nn_attribution_tables)))
    if len(nn_attribution_tables) == 1:
        axes = [axes]
    for ax, table in zip(axes, nn_attribution_tables):
        sns.barplot(data=table, x='importance_mean', y='feature', ax=ax, color='tab:green')
        freq_lbl = table['frequency'].iloc[0]
        mname_lbl = table['model'].iloc[0]
        method_lbl = table['method'].iloc[0]
        ax.set_title(f'{freq_lbl.title()} {method_lbl}: {mname_lbl} (last WF fold)')
        ax.set_xlabel('Mean |attribution| per scaled exogenous feature')
        ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print('No NN models were available for attribution.')


## Error Analysis

Residual analysis covers the best Gradient Boosting model (`best_gb`) and the GRU (`best_nn`) per frequency: residual shape, temporal behavior of forecasts, and how errors compare during normal periods versus sharp BTC growth or decline (compared against the `Naive last value` baseline).


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf


def residual_frame(frequency, model_name):
    """Остатки из OOF-прогнозов walk-forward в пространстве ДОХОДНОСТИ.

    actual/forecast - реализованная и прогнозная лог-доходность; residual =
    actual - forecast; pct_return - сама реализованная доходность (для сегментации
    шоков в анализе ниже).
    """
    oof_pred = prediction_store[frequency][model_name]['test_pred'].sort_index().dropna()
    actual = wf_frames[frequency]['BTC_logret'].reindex(oof_pred.index)
    frame = pd.DataFrame({'actual': actual, 'forecast': oof_pred}).dropna()
    frame['residual'] = frame['actual'] - frame['forecast']
    frame['abs_error'] = frame['residual'].abs()
    frame['pct_return'] = frame['actual']
    return frame


error_analysis_rows = pd.concat([best_gb, best_nn]).reset_index(drop=True)

fig, axes = plt.subplots(len(error_analysis_rows), 4, figsize=(22, 4 * len(error_analysis_rows)))
if len(error_analysis_rows) == 1:
    axes = axes.reshape(1, -1)

for row_idx, (_, row) in enumerate(error_analysis_rows.iterrows()):
    frequency = row['frequency']
    model_name = row['model']
    frame = residual_frame(frequency, model_name)

    # (1) Прогноз vs факт ДОХОДНОСТИ: scatter с линией 45°. На уровне цены такой
    # график обманчиво идеален (прогноз ~ вчерашняя цена); на доходностях он честно
    # показывает наличие/отсутствие направленного навыка (облако вокруг 45° = навык,
    # вертикальное облако у нуля = модель почти всегда предсказывает ~0).
    ax = axes[row_idx, 0]
    ax.scatter(frame['actual'], frame['forecast'], s=6, alpha=0.3, color='tab:blue')
    lim = np.nanpercentile(np.abs(frame[['actual', 'forecast']].values), 99)
    ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1)
    ax.axhline(0, color='grey', lw=0.6)
    ax.axvline(0, color='grey', lw=0.6)
    da = float(np.mean((frame['actual'] > 0) == (frame['forecast'] > 0)))
    ax.set_title(f'{frequency.title()} {model_name}\npred vs actual return (DA={da:.3f})')
    ax.set_xlabel('actual log-return')
    ax.set_ylabel('predicted')

    # (2) Ряд остатков во времени
    frame['residual'].plot(ax=axes[row_idx, 1], color='tab:red', linewidth=0.8)
    axes[row_idx, 1].axhline(0, color='black', linewidth=0.8)
    axes[row_idx, 1].set_title('residuals over time (OOF)')
    axes[row_idx, 1].set_xlabel('')

    # (3) Распределение остатков
    sns.histplot(frame['residual'], bins=40, kde=True, ax=axes[row_idx, 2], color='tab:purple')
    axes[row_idx, 2].set_title('residual distribution')

    # (4) ACF остатков: значимая автокорреляция => модель оставила невыбранную
    # предсказуемую структуру (неполная спецификация). Белый шум => спецификация ок.
    max_lags = max(1, min(30, len(frame) // 2 - 1))
    plot_acf(frame['residual'].dropna(), ax=axes[row_idx, 3], lags=max_lags)
    axes[row_idx, 3].set_title('residual ACF')

plt.tight_layout()
plt.show()


In [ ]:
# Shock-period error analysis in RETURN space.
# Сегменты OOF по величине реализованной доходности: |r| > k*std(r), k=2 (основной)
# и k=3 (проверка чувствительности). Для лучших GB/NN и бэйзлайнов считаем
# MAE/MASE/Directional Accuracy по сегментам "резкий рост"/"резкое падение"/"норма".
SHOCK_K_VALUES = [2, 3]

baseline_wf = wf_agg[wf_agg['family'] == 'baseline']
shock_candidates = pd.concat([baseline_wf, best_gb, best_nn]).reset_index(drop=True)

shock_rows = []
shock_detail_rows = []

for k in SHOCK_K_VALUES:
    for _, row in shock_candidates.iterrows():
        frequency = row['frequency']
        model_name = row['model']
        frame = residual_frame(frequency, model_name).dropna(subset=['pct_return'])
        if frame.empty:
            continue

        ret_std = frame['pct_return'].std()
        threshold = k * ret_std
        frame = frame.copy()
        frame['shock_type'] = np.where(
            frame['pct_return'] >= threshold, 'sharp growth',
            np.where(frame['pct_return'] <= -threshold, 'sharp drop', 'normal'),
        )

        oof_start_pos = wf_frames[frequency].index.get_loc(frame.index[0])
        y_train_proxy = wf_frames[frequency]['BTC_logret'].iloc[:oof_start_pos].to_numpy(dtype=float)

        for shock_type, group in frame.groupby('shock_type'):
            metrics = regression_metrics(
                group['actual'].values, group['forecast'].values,
                y_train_proxy if len(y_train_proxy) > 1 else group['actual'].values,
            )
            shock_rows.append({
                'k': k, 'frequency': frequency, 'model': model_name,
                'period_type': shock_type,
                'threshold_ret': round(threshold, 5),
                'n_observations': len(group),
                'MAE': metrics['MAE'], 'MASE': metrics['MASE'], 'DA': metrics['DA'],
            })

        if k == 2:
            shocks = frame[frame['shock_type'] != 'normal'].copy()
            shocks['abs_ret'] = shocks['pct_return'].abs()
            for time, values in shocks.sort_values('abs_ret', ascending=False).head(10).iterrows():
                shock_detail_rows.append({
                    'frequency': frequency, 'model': model_name, 'time': time,
                    'shock_type': values['shock_type'], 'realized_return': values['pct_return'],
                    'actual_ret': values['actual'], 'forecast_ret': values['forecast'],
                    'abs_error': values['abs_error'],
                })

shock_summary = pd.DataFrame(shock_rows)
shock_details = pd.DataFrame(shock_detail_rows)

print('Shock-period error analysis (|return| > 2*std), return space:')
display(shock_summary[shock_summary['k'] == 2].drop(columns='k')
        .round({'MAE': 6, 'MASE': 4, 'DA': 4, 'threshold_ret': 5}))

print('\nSensitivity check (|return| > 3*std):')
display(shock_summary[shock_summary['k'] == 3].drop(columns='k')
        .round({'MAE': 6, 'MASE': 4, 'DA': 4, 'threshold_ret': 5}))

print('\nTop individual shocks per frequency/model (k=2):')
display(shock_details.round({'realized_return': 5, 'actual_ret': 5, 'forecast_ret': 5, 'abs_error': 5}))


## Discussion

**Постановка задачи.** Все модели прогнозируют одношаговую лог-доходность
`r_t = log(P_t) - log(P_{t-1})`, а не уровень цены. Этот выбор устраняет три артефакта
уровневой постановки: (1) неспособность деревьев экстраполировать растущий тренд
(бустинги ограничены диапазоном таргета на трейне и систематически недооценивали в
поздних фолдах, из-за чего сравнение GB vs NN было некорректным); (2) вырождение SMAPE
на уровне до персистентности; (3) завышение MASE из-за смены ценового режима между
трейном и тестом. На стационарном таргете доходности GB и NN сравниваются на равных, а
уровень цены восстанавливается как `P_t = P_{t-1}·exp(r_t)`.

**Защита от утечки.** Зазор «признаки предшествуют таргету» обеспечивается на двух
уровнях: экзогенные ряды лагируются ещё в `get_data` (несинхронные сессии, лаг
публикации макро), а контемпоранные BTC-производные (RSI/MACD/ATR/OBV, скользящие
статистики) и признаки доходности строятся строго лагированными в `prep_returns`. Отбор
признаков (top-k по |Pearson|), импьютинг, клиппинг и скейлинг переобучаются внутри
каждого тренировочного фолда; гиперпараметры подбираются Optuna во вложенном
`TimeSeriesSplit`/валидационном хвосте. Интерпретация (permutation / SHAP / Captum)
считается на тех же сдвинутых фреймах и тех же моделях, метрики которых докладываются, —
рассогласования «оцениваем одно, интерпретируем другое» больше нет.

**Метрики.** Ведущая метрика — Directional Accuracy: на почти-случайном блуждании цены
MAE/RMSE доходности у всех моделей близки, и только доля верного знака отвечает на
вопрос о наличии торгуемого навыка. MASE сравнивает с наивным (нулевым) прогнозом
доходности; SMAPE приводится лишь на восстановленной цене ради сопоставимости с прежним
отчётом. Финальный арбитр практической ценности — экономический бэктест против Buy&Hold
с учётом комиссий (0.05% мейкер) и проскальзывания (2 б.п.); Sharpe считается на
процентных доходностях, а не на USD-PnL.

**Сравнение архитектур.** Вывод опирается на таблицы Directional Accuracy и бэктеста
выше. Если ни одна модель устойчиво не превышает DA ≈ 0.5 и не превосходит Buy&Hold
после издержек, корректный научный вывод — отсутствие устойчивого направленного навыка
на данных горизонтах (что согласуется со слабой формой эффективности крипторынка), а не
превосходство какой-либо одной архитектуры. Любое преимущество следует трактовать в
терминах DA/Sharpe относительно бэйзлайнов, а не низкого SMAPE на уровне цены.


## Conclusion

* Пайплайн прогнозирует одношаговую **лог-доходность** BTC на трёх частотах (дневная,
  часовая, 5-минутная) и честно сравнивает 4 бустинга (HistGB, LightGBM, XGBoost,
  CatBoost) и 4 нейросетевые архитектуры (LSTM, GRU, StackedLSTM, CNN-LSTM) в
  walk-forward (expanding для дневных, rolling для внутридневных, по 5 фолдов) с
  переподбором гиперпараметров Optuna на каждом фолде.
* Переход от уровня цены к доходности снял артефакт неспособности деревьев
  экстраполировать тренд и сделал метрики содержательными; ведущей метрикой выбрана
  **Directional Accuracy**, практическим критерием — экономический бэктест против
  **Buy&Hold** с комиссиями и проскальзыванием.
* Все шаги препроцессинга и отбора признаков выполняются строго внутри тренировочных
  фолдов, а интерпретация согласована с оцениваемыми моделями — методологических утечек
  «из будущего» в оцениваемом контуре нет.
* Лучшая модель каждого семейства и итоговый вывод о наличии/отсутствии торгуемого
  навыка определяются таблицами Directional Accuracy и экономического бэктеста выше.
* Дальнейшая работа: отбор признаков полностью внутри фолда (без глобального
  преселекта пула), вероятностный/квантильный прогноз распределения доходности,
  мультистеп-горизонты и режимно-зависимые торговые правила.
